# Lenovo AI Infrastructure Recommendation UI

This Google Colab notebook builds a web-based user interface for the pretrained infrastructure recommendation model.

The UI allows the user to:

1. Select a target Lenovo validated configuration.
2. Enter natural-language AI workload requirements.
3. Extract structured workload features.
4. Estimate the required GPU demand.
5. Calculate the total number of servers required for the selected configuration.
6. Optionally use a pretrained TensorFlow/Keras model to recommend the best configuration.

Validated configurations:

- **221 SR650**: 2x NVIDIA RTX PRO 4500 Blackwell Server Edition GPUs
- **221 SR675**: 2x NVIDIA RTX PRO 6000 Blackwell Server Edition GPUs
- **285 SR675**: 8x NVIDIA H200 GPUs
- **289 SR680**: 8x NVIDIA B200 GPUs

In [ ]:

!pip install -q gradio joblib openpyxl
!pip install -q groq

from groq import Groq
from google.colab import userdata
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import math
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import gradio as gr

from google.colab import drive
from google.colab import userdata

In [ ]:
GROQ_API_KEY = "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"

groq_client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def extract_features_with_groq_llm(user_input):
    system_prompt = """
You are an expert AI infrastructure architect.

Return ONLY valid JSON.

Use exactly these fields:
{
  "industry": "",
  "use_case": "",
  "budget_sensitivity": "",
  "workload_intensity_score": 0,
  "workload_type": "",
  "model_family": "",
  "model_parameter_size_b": 0,
  "concurrent_users": 0,
  "input_tokens": 0,
  "output_tokens": 0,
  "batch_size": 0,
  "latency_requirement": "",
  "memory_requirement_gb": 0,
  "deployment_type": "",
  "preferred_gpu_family": "",
  "storage_requirement_tb": 0,
  "network_requirement_gbps": 0,
  "business_criticality": ""
}

Allowed values:
- workload_type: inference, training, fine_tuning, rag, agentic_ai, computer_vision
- model_family: llm, computer_vision, diffusion, tabular_ml, general_ai
- latency_requirement: low, medium, high
- deployment_type: edge, enterprise, data_center
- preferred_gpu_family: RTX PRO 4500 BSE, RTX PRO 6000 BSE, H200, B200, unknown
- business_criticality: low, medium, high
- budget_sensitivity: low, medium, high

Infer missing values cautiously.
"""

    user_prompt = f"""
Extract the structured AI workload features from this user request:

{user_input}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    return json.loads(content)

In [ ]:
def align_features_to_training_schema(features):
    aligned = features.copy()

    for key, value in list(aligned.items()):
        if isinstance(value, str):
            aligned[key] = value.strip()

    if "industry" not in aligned or aligned["industry"] in [None, "", "unknown"]:
        text = json.dumps(features).lower()

        if any(x in text for x in ["hospital", "doctor", "physician", "patient", "clinical", "healthcare", "medical", "radiology"]):
            aligned["industry"] = "healthcare"
        elif any(x in text for x in ["bank", "fraud", "finance", "financial", "insurance", "trading"]):
            aligned["industry"] = "financial_services"
        elif any(x in text for x in ["retail", "store", "shopping", "ecommerce", "consumer"]):
            aligned["industry"] = "retail"
        elif any(x in text for x in ["factory", "manufacturing", "production", "industrial"]):
            aligned["industry"] = "manufacturing"
        elif any(x in text for x in ["government", "public sector", "citizen", "federal"]):
            aligned["industry"] = "government"
        elif any(x in text for x in ["telecom", "carrier", "network operator"]):
            aligned["industry"] = "telecommunications"
        else:
            aligned["industry"] = "general_enterprise"

    if "use_case" not in aligned or aligned["use_case"] in [None, "", "unknown"]:
        text = json.dumps(features).lower()

        if "rag" in text or "retrieval" in text:
            aligned["use_case"] = "rag_assistant"
        elif "fraud" in text:
            aligned["use_case"] = "fraud_detection"
        elif "vision" in text or "image" in text or "camera" in text:
            aligned["use_case"] = "computer_vision"
        elif "agent" in text:
            aligned["use_case"] = "agentic_ai_workflow"
        elif "training" in text:
            aligned["use_case"] = "model_training"
        elif "fine_tuning" in text or "fine-tuning" in text:
            aligned["use_case"] = "model_fine_tuning"
        else:
            aligned["use_case"] = "enterprise_ai_workload"

    if "budget_sensitivity" not in aligned or aligned["budget_sensitivity"] in [None, "", "unknown"]:
        text = json.dumps(features).lower()

        if any(x in text for x in ["low cost", "cost sensitive", "budget constrained", "cheap", "entry level"]):
            aligned["budget_sensitivity"] = "high"
        elif any(x in text for x in ["premium", "best performance", "maximum performance", "mission critical"]):
            aligned["budget_sensitivity"] = "low"
        else:
            aligned["budget_sensitivity"] = "medium"

    if "workload_intensity_score" not in aligned or aligned["workload_intensity_score"] in [None, "", 0, "0", "unknown"]:
        score = 0

        workload_type = aligned.get("workload_type", "")
        model_size = float(aligned.get("model_parameter_size_b", 0) or 0)
        users = int(aligned.get("concurrent_users", 0) or 0)
        latency = aligned.get("latency_requirement", "")
        memory = float(aligned.get("memory_requirement_gb", 0) or 0)

        if workload_type in ["training", "fine_tuning"]:
            score += 3
        elif workload_type in ["rag", "agentic_ai"]:
            score += 2
        else:
            score += 1

        if model_size >= 70:
            score += 3
        elif model_size >= 30:
            score += 2
        else:
            score += 1

        if users >= 500:
            score += 3
        elif users >= 100:
            score += 2
        else:
            score += 1

        if latency == "low":
            score += 1

        if memory >= 320:
            score += 1

        aligned["workload_intensity_score"] = min(score, 10)

    return aligned

## 1. Define validated Lenovo infrastructure profiles

In [ ]:
CONFIG_PROFILES = {
    "221 SR650": {
        "display_name": "Edge Configuration – 221 SR650",
        "gpu_family": "RTX PRO 4500 BSE",
        "gpus_per_server": 2,
        "description": "Edge AI, lightweight inference, small LLMs, and entry-level enterprise deployments."
    },
    "221 SR675": {
        "display_name": "Mid-Size Configuration – 221 SR675",
        "gpu_family": "RTX PRO 6000 BSE",
        "gpus_per_server": 2,
        "description": "Medium-sized inference, RAG applications, vision AI, and moderate concurrent workloads."
    },
    "285 SR675": {
        "display_name": "High-End Configuration – 285 SR675",
        "gpu_family": "H200",
        "gpus_per_server": 8,
        "description": "Large-scale inference, fine-tuning, multi-user enterprise AI, and advanced generative AI workloads."
    },
    "289 SR680": {
        "display_name": "Ultra High-End Configuration – 289 SR680",
        "gpu_family": "B200",
        "gpus_per_server": 8,
        "description": "Large-scale training, agentic AI systems, massive concurrent inference, and foundation model deployment."
    }
}

CONFIG_OPTIONS = list(CONFIG_PROFILES.keys())
AUTO_OPTION = "Auto - Use model recommendation"

## 3. Load pretrained model artifacts if available

In [ ]:
drive.mount('/content/drive')

ARTIFACT_PATH = "/content/drive/MyDrive/ColabNotebooks/Walsh/CapsProject2026"

MODEL_PATH = f"{ARTIFACT_PATH}/lenovo_server_recommender_dnn.keras"
PREPROCESSOR_PATH = f"{ARTIFACT_PATH}/preprocessor.joblib"
LABEL_ENCODER_PATH = f"{ARTIFACT_PATH}/label_encoder.joblib"

loaded_model = None
loaded_preprocessor = None
loaded_label_encoder = None
model_status = []

try:
    if os.path.exists(MODEL_PATH):
        loaded_model = tf.keras.models.load_model(MODEL_PATH)
        model_status.append("Keras DNN model loaded.")
    else:
        model_status.append("Keras model not found. Rule-based fallback enabled.")

    if os.path.exists(PREPROCESSOR_PATH):
        loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)
        model_status.append("Preprocessor loaded.")
    else:
        model_status.append("Preprocessor not found. Model prediction disabled.")

    if os.path.exists(LABEL_ENCODER_PATH):
        loaded_label_encoder = joblib.load(LABEL_ENCODER_PATH)
        model_status.append("Label encoder loaded.")
    else:
        model_status.append("Label encoder not found. Model prediction disabled.")

except Exception as e:
    model_status.append(f"Artifact loading error: {e}")

print("\n".join(model_status))

Mounted at /content/drive
Keras DNN model loaded.
Preprocessor loaded.
Label encoder loaded.


## 4. Natural-language feature extraction

This function extracts structured workload features from a user conversation using groq.



## 5. GPU demand estimator and server count calculator

The DNN predicts the best configuration class. However, to calculate the total number of servers, the application also estimates the total GPU demand for the project and divides that number by the GPUs per server in the selected configuration.

In [ ]:
TRAINING_FEATURE_COLUMNS = [
    "industry",
    "use_case",
    "budget_sensitivity",
    "workload_intensity_score",
    "workload_type",
    "model_family",
    "model_parameter_size_b",
    "concurrent_users",
    "input_tokens",
    "output_tokens",
    "batch_size",
    "latency_requirement",
    "memory_requirement_gb",
    "deployment_type",
    "preferred_gpu_family",
    "storage_requirement_tb",
    "network_requirement_gbps",
    "business_criticality"
]


def predict_configuration_with_dnn(features: dict):
    if loaded_model is None or loaded_preprocessor is None or loaded_label_encoder is None:
        return None, None

    try:
        input_df = pd.DataFrame([{col: features.get(col, None) for col in TRAINING_FEATURE_COLUMNS}])
        X_processed = loaded_preprocessor.transform(input_df)
        probabilities = loaded_model.predict(X_processed, verbose=0)[0]
        pred_idx = int(np.argmax(probabilities))
        predicted_label = loaded_label_encoder.inverse_transform([pred_idx])[0]
        confidence = float(np.max(probabilities))
        return predicted_label, confidence

    except Exception as e:
        return None, f"Prediction error: {e}"

In [ ]:
def build_recommendation(target_config, requirement_text):
    if not requirement_text or len(requirement_text.strip()) < 10:
        return "Please enter a more detailed AI workload requirement.", "", ""

    try:
        extracted_features = extract_features_with_groq_llm(requirement_text)
    except Exception as e:
        print("Groq extraction failed. Using fallback parser:", e)
        extracted_features = extract_features_from_text(requirement_text)

    features = align_features_to_training_schema(extracted_features)

    predicted_config, confidence = predict_configuration_with_dnn(features)

    if target_config == AUTO_OPTION:
        selected_config = predicted_config if predicted_config in CONFIG_PROFILES else "221 SR675"
        selection_source = "DNN model recommendation" if predicted_config else "Fallback default recommendation"
    else:
        selected_config = target_config
        selection_source = "User-selected target configuration"

    required_gpus = estimate_required_gpus(features)
    total_servers = calculate_server_count(required_gpus, selected_config)

    profile = CONFIG_PROFILES[selected_config]

    summary = f"""
## Recommendation Summary

**Selected Configuration:** {selected_config}
**Configuration Name:** {profile['display_name']}
**GPU Family:** {profile['gpu_family']}
**GPUs per Server:** {profile['gpus_per_server']}
**Estimated Required GPUs:** {required_gpus}
**Total Servers Required:** {total_servers}

**Selection Source:** {selection_source}
"""

    if predicted_config:
        summary += f"\n**DNN Model Recommended Configuration:** {predicted_config}  "
        if isinstance(confidence, float):
            summary += f"\n**Model Confidence:** {confidence:.2%}"
    else:
        summary += "\n**DNN Model Status:** Model artifacts were not loaded, so the UI used the selected configuration and rule-based GPU estimation."

    explanation = f"""
## Business and Technical Interpretation

The system estimated that this workload requires approximately **{required_gpus} GPUs** based on workload type, model size, concurrency, token volume, latency expectations, memory needs, and business criticality.

Using the selected Lenovo validated configuration **{selected_config}**, which provides **{profile['gpus_per_server']} GPUs per server**, the project requires:

# **{total_servers} server(s)**

This configuration is intended for: {profile['description']}
"""

    structured_features = json.dumps(features, indent=4)

    return summary, structured_features, explanation

In [ ]:
def estimate_required_gpus(features: dict) -> int:
    workload = features["workload_type"]
    model_size = float(features["model_parameter_size_b"])
    users = int(features["concurrent_users"])
    input_tokens = int(features["input_tokens"])
    output_tokens = int(features["output_tokens"])
    batch_size = int(features["batch_size"])
    latency = features["latency_requirement"]
    memory_gb = float(features["memory_requirement_gb"])
    criticality = features["business_criticality"]

    if workload == "training":
        if model_size >= 100:
            base_gpus = 16
        elif model_size >= 70:
            base_gpus = 8
        elif model_size >= 30:
            base_gpus = 4
        else:
            base_gpus = 2
    elif workload == "fine_tuning":
        if model_size >= 70:
            base_gpus = 8
        elif model_size >= 30:
            base_gpus = 4
        else:
            base_gpus = 2
    elif workload == "agentic_ai":
        if model_size >= 70:
            base_gpus = 8
        else:
            base_gpus = 4
    elif workload == "rag":
        if model_size >= 70:
            base_gpus = 4
        else:
            base_gpus = 2
    elif workload == "computer_vision":
        base_gpus = 2 if users <= 100 else 4
    else:
        if model_size >= 70:
            base_gpus = 4
        elif model_size >= 30:
            base_gpus = 2
        else:
            base_gpus = 1

    user_multiplier = max(1.0, users / 100)

    total_tokens = input_tokens + output_tokens
    token_multiplier = 2.0 if total_tokens >= 8000 else 1.5 if total_tokens >= 4000 else 1.0

    latency_multiplier = 1.25 if latency == "low" else 1.0
    batch_multiplier = max(1.0, batch_size / 8)

    if memory_gb >= 640:
        memory_multiplier = 2.0
    elif memory_gb >= 320:
        memory_multiplier = 1.5
    else:
        memory_multiplier = 1.0

    criticality_multiplier = 1.2 if criticality == "high" else 1.0

    estimated = (
        base_gpus
        * user_multiplier
        * token_multiplier
        * latency_multiplier
        * batch_multiplier
        * memory_multiplier
        * criticality_multiplier
    )

    return max(1, math.ceil(estimated))


def calculate_server_count(required_gpus: int, config_name: str) -> int:
    gpus_per_server = CONFIG_PROFILES[config_name]["gpus_per_server"]
    return max(1, math.ceil(required_gpus / gpus_per_server))

## 6. Optional DNN model prediction

This function attempts to use the exported TensorFlow/Keras model. If model artifacts are not available, it returns `None` and the UI will rely on the selected target configuration.

## 7. Recommendation engine

In [ ]:
test_text = "I need to deploy a manufacturing AI system to host an 80B parameter LLM serving 1000 concurrent users."

test_features = extract_features_with_groq_llm(test_text)
print(json.dumps(test_features, indent=4))

{
    "industry": "manufacturing",
    "use_case": "serving",
    "budget_sensitivity": "high",
    "workload_intensity_score": 9,
    "workload_type": "inference",
    "model_family": "llm",
    "model_parameter_size_b": 80,
    "concurrent_users": 1000,
    "input_tokens": 0,
    "output_tokens": 0,
    "batch_size": 0,
    "latency_requirement": "low",
    "memory_requirement_gb": 0,
    "deployment_type": "enterprise",
    "preferred_gpu_family": "unknown",
    "storage_requirement_tb": 0,
    "network_requirement_gbps": 0,
    "business_criticality": "high"
}


## 8. Launch Gradio web interface

In [ ]:
with gr.Blocks(title="Lenovo AI Infrastructure Recommendation Engine") as demo:
    gr.Markdown(
        """
        # Lenovo AI Infrastructure Recommendation Engine

        Enter a natural-language AI workload requirement and select a target Lenovo validated configuration.

        The system uses:
        1. **Groq Llama LLM** to extract structured workload features.
        2. **Pretrained TensorFlow/Keras DNN** to recommend the best Lenovo configuration.
        3. A server-count calculator to estimate the total number of servers required.
        """
    )

    gr.Markdown(
        f"""
        **Model Status:**
        {chr(10).join(model_status)}
        """
    )

    with gr.Row():
        target_config = gr.Dropdown(
            choices=[AUTO_OPTION] + CONFIG_OPTIONS,
            value=AUTO_OPTION,
            label="Target Configuration"
        )

    requirement_text = gr.Textbox(
        label="AI Workload Requirement",
        placeholder=(
            "Example: I need to deploy a manufacturing AI system to host an 80B parameter LLM "
            "serving 1000 concurrent users with low latency in an enterprise data center."
        ),
        lines=8
    )

    run_button = gr.Button("Generate Recommendation")

    summary_output = gr.Markdown(label="Recommendation Summary")
    features_output = gr.Code(label="Extracted Structured Features", language="json")
    explanation_output = gr.Markdown(label="Explanation")

    run_button.click(
        fn=build_recommendation,
        inputs=[target_config, requirement_text],
        outputs=[summary_output, features_output, explanation_output]
    )

demo.launch(share=True, debug=True)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://902eb2431d6ce1f74c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://902eb2431d6ce1f74c.gradio.live
Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://902eb2431d6ce1f74c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

## 9. Notes for future web deployment

For production deployment, export and package the following:

- `lenovo_server_config_dnn.keras`
- `preprocessor.pkl`
- `label_encoder.pkl`
- The feature extraction function or an LLM API wrapper
- The Gradio or Streamlit UI layer

Recommended next enhancement:

- Replace the lightweight parser with a real LLM structured JSON extraction layer.
- Add ROI and revenue impact estimation.
- Add a downloadable PDF/Excel recommendation report.